# 03 · Practice drill: the day-of flow, 10 uploads, hidden holdout

30% of the labelled practice sessions are hidden behind a mock leaderboard and written out as if HRT had shipped a new dataset. Run the real day-of command on it, then follow the upload plan (README, competition day). Uploads are scarce, so log every one, and check at the end whether the public score picked the best private one.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from arena.config import ARCADE_ALGO_ARENA, TaskSpec
from arena.data import list_data_files, load_arcade, truncate
from arena.paths import SIGNALS, SUBMISSIONS, default_data_dir

DATA_DIR = default_data_dir()  # data/raw on the day; override: DATA_DIR = Path("...")
SPEC = ARCADE_ALGO_ARENA       # CHANGE if the rules differ: halfway bar, exit bar, Sharpe factor, columns
print(DATA_DIR)
list_data_files(DATA_DIR)

In [ ]:
import tempfile

from arena.practice import MockLeaderboard, holdout_split

src = Path(DATA_DIR)
bars, heads = pd.read_parquet(src / "ins_bars.parquet"), pd.read_parquet(src / "ins_headlines.parquet")
rets = pd.read_parquet(src / "ins_returns.parquet")
ids = np.sort(bars["session"].unique())
train_idx, hidden_idx = holdout_split(len(ids), frac=0.3, seed=2026)
train_ids, hidden_ids = ids[train_idx], ids[hidden_idx]
drill = Path(tempfile.mkdtemp()) / "drill"
drill.mkdir()
bars[bars.session.isin(train_ids)].to_parquet(drill / "ins_bars.parquet", index=False)
heads[heads.session.isin(train_ids)].to_parquet(drill / "ins_headlines.parquet", index=False)
rets[rets.session.isin(train_ids)].to_parquet(drill / "ins_returns.parquet", index=False)
seen = bars.session.isin(hidden_ids) & (bars.bar_ix <= SPEC.halfway_bar)
bars[seen].to_parquet(drill / "oos_bars.parquet", index=False)
heads[heads.session.isin(hidden_ids) & (heads.bar_ix <= SPEC.halfway_bar)].to_parquet(drill / "oos_headlines.parquet", index=False)
hidden_y = rets.set_index("session")["returns"].reindex(hidden_ids).to_numpy()
lb = MockLeaderboard(hidden_ids, hidden_y, SPEC, budget=10, private_frac=0.5)
print(f"{len(train_ids)} training sessions, {len(hidden_ids)} hidden; uploads left: {lb.remaining}")

## T+1: the day-of command (prints NEW DATA here, because the drill files are a new split)

In [ ]:
from arena.runner import main as arena_run

out = drill.parent / "uploads"
arena_run(["--data", str(drill), "--extra", "none", "--out", str(out), "--cv", "none"])

## T+3 onwards: follow the upload plan, one file per upload, in order

In [ ]:
for f in sorted(out.glob("*_hrtonly.csv")):
    sub = pd.read_csv(f)
    print(f"{f.name:32s} public {lb.submit(sub[SPEC.id_col], sub[SPEC.pred_col], label=f.stem):.3f}")
lb.history()

In [ ]:
# End of the drill: did the public score pick the best private one?
lb.history(reveal_private=True)